
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


#Lecture - Introduction to Multiplex Streaming, Delta Sinks, and Iceberg Reads

This lecture introduces three advanced concepts used in Spark Declarative Pipelines: the **Multiplex pattern** for efficiently ingesting mixed event streams, **Delta Sinks** for writing streaming data to external tables, and **Iceberg reads via Delta UniForm** for enabling cross-platform access to Delta tables.
## Learning Objectives

By the end of this lecture, you will be able to:

- Explain the **Multiplex pattern** and how fan-out works for mixed event streams
- Describe what a **Delta Sink** is and when to use it over managed streaming tables
- Explain how **Delta UniForm** enables Iceberg reads without data duplication
- Understand how these three concepts **chain together** in a single pipeline

## A. The Multiplex Pattern

The Multiplex pattern addresses a common production challenge: efficiently processing multiple event types that arrive through a single data transport mechanism.

### A1. The Problem: One Stream, Many Schemas

In production environments, multiple business systems often share a **single data transport** such as:
- One Kafka topic
- One cloud storage path  
- One message queue

Each message carries a **type field** identifying which business domain it belongs to. Without the Multiplex pattern, this creates significant operational overhead:

<div style="display:flex;gap:16px;margin:16px 0;">
<div style="flex:1;background:#FFEBEE;border:1px solid #FFCDD2;border-radius:6px;padding:14px 16px;">
<div style="font-weight:bold;color:#C62828;font-size:13px;margin-bottom:8px;"> Without Multiplex</div>
<ul style="font-size:13px;color:#333;margin:0;padding-left:16px;">
<li>N event types → N separate pipelines</li>
<li>N separate checkpoints and source scans</li>
<li>Source change must be applied N times</li>
</ul>
</div>
<div style="flex:1;background:#E8F5E9;border:1px solid #C8E6C9;border-radius:6px;padding:14px 16px;">
<div style="font-weight:bold;color:#1B5E20;font-size:13px;margin-bottom:8px;"> With Multiplex</div>
<ul style="font-size:13px;color:#333;margin:0;padding-left:16px;">
<li>N event types → 1 ingestion pipeline</li>
<li>1 checkpoint, 1 source scan shared across all domains</li>
<li>Source change applied in one place</li>
</ul>
</div>
</div>


**Reference Documentation:**
- [Multiplexing Data Pipelines Blog](https://www.databricks.com/blog/2022/04/27/how-uplift-built-cdc-and-multiplexing-data-pipelines-with-databricks-delta-live-tables.html)

### A2. Ingest Once, Fan Out by Type

The Multiplex pattern follows a simple but powerful approach:

1. **Single Ingestion**: Read all event types into one bronze table
2. **Type-Based Filtering**: Use the event type field to separate domains
3. **Fan-Out Processing**: Create domain-specific tables downstream

**Architecture Flow:**
<div class="mermaid">
flowchart LR
    SRC["☁️ Mixed Event Stream"]
    BRZ["Bronze Table
All event types together
VARIANT PAYLOAD"]
    A["Domain A Table
WHERE type = A"]
    B["Domain B Table
WHERE type = B"]
    C["Domain C Table
WHERE type = C"]
    SRC -->|"Single ingest
one checkpoint"| BRZ
    BRZ --> A
    BRZ --> B
    BRZ --> C
    style SRC fill:#607D8B,color:#fff
    style BRZ fill:#FF5722,color:#fff
    style A fill:#EF6C00,color:#fff
    style B fill:#EF6C00,color:#fff
    style C fill:#EF6C00,color:#fff
</div>
<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>

**Key Benefits:**
- Single checkpoint management
- Shared source scanning
- Centralized error handling
- Simplified monitoring

## B. Sinks

Sinks provide a mechanism to write streaming data from a Spark Declarative Pipeline to external Delta tables that exist outside the pipeline's managed scope.

<div style="display:flex;justify-content:left;margin:32px 0;">
<div class="mermaid" style="min-width:750px;">
flowchart LR
    F["Pipeline Flow\nappend_flow"]
    D["Default"]
    A["@dp.append_flow"]
    subgraph PIPE["   Inside Pipeline — Managed Scope   "]
        ST["Streaming Table\nMaterialized View"]
    end
    subgraph EXT["   Outside Pipeline — External Targets   "]
        SK["Sink\nDelta · Kafka · Custom"]
    end
    F --- D --> ST
    F --- A --> SK
    style F fill:#FF7043,color:#fff,stroke:#E64A19,stroke-width:2px
    style D fill:#F5F5F5,color:#333,stroke:#BDBDBD,stroke-width:1px
    style A fill:#F5F5F5,color:#333,stroke:#BDBDBD,stroke-width:1px
    style ST fill:#FFF3E0,stroke:#F57C00,stroke-width:2px,color:#000
    style SK fill:#1565C0,color:#fff,stroke:#0D47A1,stroke-width:2px
    style PIPE fill:#FFF8E1,stroke:#FF9800,stroke-width:2px,color:#E65100
    style EXT fill:#E3F2FD,stroke:#1976D2,stroke-width:2px,color:#0D47A1
</div>
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({
    startOnLoad: true,
    theme: "default",
    flowchart: { padding: 40, nodeSpacing: 60, rankSpacing: 80 }
});
</script>

<div style="background:#F8F9FA;border-left:4px solid #607D8B;padding:14px 18px;margin:24px 0;border-radius:4px;">
  <div style="font-size:16px;color:#444;line-height:1.8;">
    <strong>Only the Python API is supported</strong> — SQL is not supported for sinks. Only <code>append_flow</code> can write to a sink.
  </div>
</div>

**Reference Documentation:**
- [Sinks in SDP](https://docs.databricks.com/delta-live-tables/dlt-sinks.html)

### B1. Supported Sink Types

Databricks supports <strong>four types of sinks</strong> — each suited for a different destination and use case.


<div style="display:flex;gap:0;margin:24px 0;align-items:stretch;">

  <div style="flex:1;background:#E8F5E9;border-top:4px solid #388E3C;border-radius:12px 0 0 12px;padding:24px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:14px;">Delta Table Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Unity Catalog managed tables</li>
      <li>External Delta tables</li>
      <li>Write by path or table name</li>
    </ul>
  </div>

  <div style="flex:1;background:#E3F2FD;border-top:4px solid #1976D2;padding:24px;">
    <div style="font-weight:bold;color:#0D47A1;font-size:16px;margin-bottom:14px;">Apache Kafka Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Write back to Kafka topics</li>
      <li>Low-latency operational use cases</li>
      <li>Reverse ETL out of Databricks</li>
    </ul>
  </div>

  <div style="flex:1;background:#FFF3E0;border-top:4px solid #F57C00;padding:24px;">
    <div style="font-weight:bold;color:#E65100;font-size:16px;margin-bottom:14px;">Azure Event Hubs Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Uses Kafka interface format</li>
      <li>Real-time event streaming</li>
      <li>Fraud detection · recommendations</li>
    </ul>
  </div>

  <div style="flex:1;background:#F3E5F5;border-top:4px solid #7B1FA2;border-radius:0 12px 12px 0;padding:24px;">
    <div style="font-weight:bold;color:#6A1B9A;font-size:16px;margin-bottom:14px;">Python Custom Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Write to any data store</li>
      <li>Uses PySpark custom data sources</li>
      <li>Maximum flexibility</li>
    </ul>
  </div>

</div>

<div style="background:#E8F5E9;border-left:4px solid #388E3C;padding:14px 18px;margin:20px 0;border-radius:4px;">
  <div style="font-size:16px;color:#1B5E20;line-height:1.8;">
    For code examples of each sink type, refer to the <a href="https://docs.databricks.com/aws/en/ldp/ldp-sinks?language=Delta%C2%A0sinks#create-a-sink" style="color:#1976D2;">Creating a Sink Documentation</a>
  </div>
</div>


### B2. Managed Tables vs. Sinks

Every standard dataset in a Spark Declarative Pipeline — streaming table or materialized view — is **owned and managed by the pipeline**. A **sink** breaks this intentionally: it lets the pipeline write streaming data to a **plain Delta table that exists outside the pipeline's managed scope**.

<div style="display:flex;gap:0;margin:24px 0;align-items:stretch;">

  <div style="flex:1;background:#E8F5E9;border-top:4px solid #388E3C;border-radius:12px 0 0 12px;padding:24px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:14px;">Managed Table (Default)</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Data stays within Unity Catalog</li>
      <li>Full pipeline lineage tracking</li>
      <li>Supports expectations and CDC</li>
      <li>Streaming Tables and Materialized Views</li>
    </ul>
  </div>

  <div style="display:flex;align-items:center;justify-content:center;padding:0 20px;background:#F8F9FA;border-top:1px solid #E0E0E0;border-bottom:1px solid #E0E0E0;">
    <div style="font-size:24px;color:#999;font-weight:300;">vs</div>
  </div>

  <div style="flex:1;background:#E3F2FD;border-top:4px solid #1976D2;border-radius:0 12px 12px 0;padding:24px;">
    <div style="font-weight:bold;color:#0D47A1;font-size:16px;margin-bottom:14px;">Sink</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Write to external systems outside Databricks</li>
      <li>Enables reverse ETL and operational use cases</li>
      <li>Supports Kafka, Event Hubs, custom targets</li>
      <li>No expectations — append only</li>
    </ul>
  </div>

</div>


### B3. Delta Sink In Action


A <strong>Delta sink</strong> writes pipeline output to a Delta table <em>outside</em> the pipeline's managed lifecycle — unlocking configurations not possible on pipeline-managed streaming tables, such as <strong>Iceberg compatibility</strong>.

---


<div style="display:flex;gap:16px;margin:16px 0 28px 0;">
  <div style="flex:1;background:#FFF3E0;border-left:4px solid #F57C00;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#E65100;font-size:16px;margin-bottom:8px;">Streaming Tables Cannot</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Enable Iceberg compatibility</li>
      <li>Share with non-Databricks platforms</li>
    </ul>
  </div>
  <div style="flex:1;background:#E8F5E9;border-left:4px solid #388E3C;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:8px;">Delta Sink Can</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Have full Delta table property control</li>
      <li>Enables Iceberg UniForm for cross-platform reads</li>
    </ul>
  </div>
</div>

---

#### Implementation

<div style="display:flex;gap:0;margin:20px 0;align-items:stretch;">

  <div style="flex:1;background:#E8F5E9;border-top:4px solid #388E3C;border-radius:12px 0 0 12px;padding:24px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:16px;">Step 1 — Register the Sink</div>
    <pre style="background:#F8F9FA;border:1px solid #E0E0E0;border-radius:6px;padding:16px;font-size:14px;line-height:1.8;overflow-x:auto;">
<div class="code-block" data-language="python">
from pyspark import pipelines as dp

dp.create_sink(
    name    = "my_sink",
    format  = "delta",
    options = {
        "tableName": "catalog.schema.table"
    }
)</pre>
  </div>

  <div style="display:flex;align-items:center;justify-content:center;padding:0 20px;background:#F8F9FA;border-top:1px solid #E0E0E0;border-bottom:1px solid #E0E0E0;">
    <div style="font-size:24px;color:#999;font-weight:300;">→</div>
  </div>

  <div style="flex:1;background:#E3F2FD;border-top:4px solid #1976D2;border-radius:0 12px 12px 0;padding:24px;">
    <div style="font-weight:bold;color:#0D47A1;font-size:16px;margin-bottom:16px;">Step 2 — Write to the Sink</div>
    <pre style="background:#F8F9FA;border:1px solid #E0E0E0;border-radius:6px;padding:16px;font-size:14px;line-height:1.8;overflow-x:auto;">

<div class="code-block" data-language="python">
@dp.append_flow(
name   = "my_sink_flow",
target = "my_sink"
)
def my_sink_flow():
    return spark.readStream.table(
        "schema.source_table"
    )</pre>
  </div>

</div>

<div style="background:#F8F9FA;border-left:4px solid #607D8B;border-radius:4px;padding:16px 20px;margin-top:8px;">
  <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
    <li>Checkpointing is handled automatically by <code>append_flow</code></li>
    <li>Only new records written per run — no overwrites</li>
    <li>Python only — no SQL equivalent</li>
  </ul>
</div>
<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'sql';
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML = 
            '<div style="position:relative;margin:16px 0;">' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:40px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:14px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## C. Iceberg Reads via Delta UniForm

Delta UniForm enables cross-platform access to Delta tables by automatically generating Apache Iceberg metadata without duplicating the underlying data.

### C1. The Cross-Platform Challenge

<div style="font-size:16px;color:#555;line-height:1.8;margin-bottom:24px;">
  Modern data architectures span multiple platforms. Traditionally, supporting each platform meant <strong>copying data into multiple formats</strong> — creating duplication, sync issues, and increased storage costs. <strong>Delta UniForm</strong> solves this with a single set of files and two metadata layers.
</div>

<div style="display:flex;gap:24px;margin:24px 0;align-items:flex-start;">

  <div style="flex:1;display:flex;flex-direction:column;align-items:center;">
    <div class="mermaid" style="width:100%;">
flowchart TB
    subgraph OLD["Traditional Approach"]
        D1["Delta Files"] --> C["Copy and Convert"]
        C --> IC["Iceberg Copy"]
        C --> PA["Parquet Copy"]
        C --> HV["Hive Copy"]
    end
    style D1 fill:#FFEBEE,stroke:#C62828,color:#000
    style C fill:#FFCDD2,stroke:#C62828,color:#000
    style IC fill:#FFEBEE,stroke:#C62828,color:#000
    style PA fill:#FFEBEE,stroke:#C62828,color:#000
    style HV fill:#FFEBEE,stroke:#C62828,color:#000
    style OLD fill:#FFF3F3,stroke:#C62828,stroke-width:2px,color:#C62828
    </div>
  </div>

  <div style="display:flex;align-items:center;justify-content:center;padding:0 8px;margin-top:80px;">
    <div style="font-size:28px;color:#999;font-weight:300;"></div>
  </div>

  <div style="flex:1;display:flex;flex-direction:column;align-items:center;">
    <div class="mermaid" style="width:100%;">
flowchart TB
    subgraph NEW["Delta UniForm"]
        PQ["One Set of Parquet Files"] --> DM["Delta Metadata"]
        PQ --> IM["Iceberg Metadata"]
    end
    style PQ fill:#E8F5E9,stroke:#388E3C,color:#000
    style DM fill:#E3F2FD,stroke:#1976D2,color:#000
    style IM fill:#E3F2FD,stroke:#1976D2,color:#000
    style NEW fill:#F1F8E9,stroke:#388E3C,stroke-width:2px,color:#388E3C
    </div>
  </div>

</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: true, theme: "default" });
</script>

<div style="display:flex;gap:16px;margin:24px 0;">
  <div style="flex:1;background:#FFEBEE;border-left:4px solid #C62828;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#C62828;font-size:16px;margin-bottom:8px;">Traditional Approach</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>Data copied into multiple formats</li>
      <li>Sync issues across copies</li>
      <li>Increased storage costs</li>
    </ul>
  </div>
  <div style="flex:1;background:#E8F5E9;border-left:4px solid #388E3C;border-radius:4px;padding:16px 20px;">
    <div style="font-weight:bold;color:#1B5E20;font-size:16px;margin-bottom:8px;">Delta UniForm</div>
    <ul style="font-size:16px;color:#555;line-height:2;margin:0;padding-left:20px;">
      <li>One set of Parquet files</li>
      <li>Delta + Iceberg metadata layers</li>
      <li>No duplication — automatic sync</li>
    </ul>
  </div>
</div>

<div style="background:#E3F2FD;border-left:4px solid #1976D2;padding:14px 18px;border-radius:4px;">
  <div style="font-size:16px;color:#0D47A1;line-height:1.8;">
    For full details refer to the <a href="https://docs.databricks.com/aws/en/delta/uniform" style="color:#1976D2;font-weight:bold;">Delta UniForm Documentation →</a>
  </div>
</div>


### C2. How Delta UniForm Works

  Delta UniForm maintains <strong>one physical dataset</strong> with <strong>two logical views</strong> — no data duplication, no separate copies. Iceberg metadata is generated <strong>asynchronously</strong> after every Delta write, keeping both views in sync automatically.


---
<div class="mermaid" style="min-width:700px;">
flowchart LR
    subgraph STORE["<b>One Delta Table — One Set of Parquet Files</b>"]
        PF("<div style='text-align:center;font-size:13px;'>
<img src='https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/icons/file_icon.png' width='10' style='display:inline-block;margin:0 3px;'/>
<img src='https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/icons/file_icon.png' width='10' style='display:inline-block;margin:0 3px;'/>
<br/><b>Parquet Data Files</b>
</div>")
        DL["Delta Transaction Log\n_delta_log/"]
        IM["Iceberg Metadata\nmetadata/*.metadata.json"]
        PF --- DL
        PF --- IM
    end
    DC["Databricks Clients\nRead via Delta protocol"]
    IC["External Tools\nSnowflake · Trino · Athena · Spark OSS\nRead via Iceberg REST Catalog"]
    DL --> DC
    IM --> IC
    style DL fill:#1565C0,color:#fff,stroke:#0D47A1,stroke-width:2px
    style IM fill:#2E7D32,color:#fff,stroke:#1B5E20,stroke-width:2px
    style DC fill:#FF3621,color:#fff,stroke:#C62828,stroke-width:2px
    style IC fill:#455A64,color:#fff,stroke:#263238,stroke-width:2px
    style STORE fill:#F8F9FA,stroke:#607D8B,stroke-width:2px,color:#333
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({
    startOnLoad: true,
    theme: "default",
    flowchart: { padding: 40, nodeSpacing: 60, rankSpacing: 80 }
});
</script>
### Enabling Iceberg Reads 

<div style="display:flex;flex-direction:column;gap:10px;margin:16px 0;">
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">1️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Disable Deletion Vectors</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.enableDeletionVectors' = 'false'</code><br>
        Iceberg v2 cannot represent Delta's soft-delete markers. Disabling ensures all deletes are hard deletes, making the table fully readable by Iceberg clients.
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">2️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Column Mapping Mode</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.columnMapping.mode' = 'name'</code><br>
        Ensures column identifiers are consistent between Delta and Iceberg schemas, preventing schema drift and enabling seamless cross-platform access.
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">3️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Enable IcebergCompatV2</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.enableIcebergCompatV2' = 'true'</code><br>
        Activates Delta's write protocol compatible with Iceberg v2, allowing Iceberg clients to read Delta tables without data conversion.
      </div>
    </div>
  </div>
  <div style="display:flex;align-items:flex-start;gap:12px;background:#F5F5F5;border-radius:6px;padding:12px 14px;">
    <div style="font-size:22px;min-width:32px;">4️⃣</div>
    <div>
      <div style="font-weight:bold;font-size:13px;color:#333;">Enable Universal Format</div>
      <div style="font-size:13px;color:#555;margin-top:2px;">
        <strong>Property:</strong> <code style="font-size:12px;">'delta.universalFormat.enabledFormats' = 'iceberg'</code><br>
        Triggers asynchronous Iceberg metadata generation after every Delta commit, ensuring up-to-date Iceberg views for external tools.
      </div>
    </div>
  </div>
</div>

<div style="background:#FFEBEE;border-left:4px solid #F44336;border-radius:4px;padding:14px 18px;margin:16px 0;">
  <div style="font-weight:bold;font-size:14px;color:#B71C1C;margin-bottom:6px;">⚠️ Iceberg Reads Only Work on Plain Delta Tables</div>
  <div style="font-size:14px;color:#333;">
    <strong>Note:</strong> Pipeline-managed <b>streaming tables</b> and <b>materialized views</b> cannot have Iceberg reads enabled. Only a plain external Delta table—such as one created via a <b>Delta Sink</b>—supports UniForm. This bridge is required for cross-platform access.
    <br><br>
    <strong>Additional Details:</strong> Setting these properties is a one-time operation, but must be done before any data is written. Once enabled, Iceberg metadata is kept in sync automatically, allowing tools like Snowflake, Trino, and Athena to query the same Delta table.
  </div>
</div>

---

<div style="background:#E3F2FD;border-left:4px solid #1976D2;padding:14px 18px;border-radius:4px;margin-top:8px;">
  <div style="font-size:16px;color:#0D47A1;line-height:1.8;">
    For more details refer to the <a href="https://docs.databricks.com/aws/en/delta/uniform" style="color:#1976D2;font-weight:bold;">Delta UniForm Documentation</a>
  </div>
</div>


#### EXPAND FOR ADDITIONAL DETAILS
<details>


<div style="font-size:16px;color:#555;line-height:2;margin-bottom:24px;">

Delta UniForm (Universal Format) solves a long-standing problem in the lakehouse ecosystem: <strong>different tools speak different table formats</strong>. Snowflake, Trino, Athena, and open-source Spark all understand Apache Iceberg — but not Delta Lake natively. UniForm bridges this gap by making your Delta tables <em>simultaneously readable as Iceberg tables</em>, without any data duplication or separate pipelines.

</div>

<div style="font-size:16px;color:#555;line-height:2;margin-bottom:24px;">

Both Delta Lake and Apache Iceberg are built on the same foundation: <strong>Parquet data files</strong> plus a <strong>metadata layer</strong>. UniForm exploits this by asynchronously generating Iceberg metadata after every Delta write — the same Parquet files now serve two formats at once. From a Delta client's perspective, nothing changes. From an Iceberg client's perspective, it looks like a native Iceberg table. You get <strong>one dataset, two logical views, zero duplication</strong>.

</div>

---

#### Benefits

<div style="font-size:16px;color:#555;line-height:2;">

- <strong>No data movement:</strong> A single copy of Parquet files is shared across Delta and Iceberg clients — no ETL, no replication, no storage overhead.
- <strong>Cross-platform interoperability:</strong> Tools like Snowflake, Trino, Apache Athena, and open-source Spark can query your Delta tables directly via the Iceberg REST Catalog.
- <strong>Negligible write overhead:</strong> Iceberg metadata generation happens asynchronously after the Delta commit, so your write pipelines are not slowed down.
- <strong>Works with Unity Catalog:</strong> Unity Catalog acts as the Iceberg REST Catalog, meaning external clients get governed, catalogued access without any extra infrastructure.
- <strong>Performance parity:</strong> Benchmarks show comparable read performance between Delta UniForm and natively managed Iceberg tables on Snowflake.

</div>

---

#### Limitations

<div style="font-size:16px;color:#555;line-height:2;">

- <strong>Read-only for Iceberg clients:</strong> External tools can only <em>read</em> via Iceberg. All writes must go through Delta — you cannot write to a UniForm table from Snowflake or Trino.
- <strong>Deletion Vectors must be disabled:</strong> Tables using deletion vectors (a Delta performance feature) cannot have UniForm enabled. You must set <code>'delta.enableDeletionVectors' = 'false'</code>.
- <strong>Streaming Tables and Materialized Views are excluded:</strong> Pipeline-managed tables in Spark Declarative Pipelines cannot have the required <code>delta.universalFormat.enabledFormats</code> property set directly. This is where <strong>Delta Sinks</strong> come in — they write to plain external Delta tables where UniForm can be freely enabled.
- <strong>Column mapping is permanent:</strong> Enabling UniForm also enables column mapping (<code>delta.columnMapping.mode = 'name'</code>), which cannot be dropped once set.
- <strong>Databricks Runtime 14.3 LTS or above required:</strong> Any Databricks client writing to a UniForm-enabled table must use DBR 14.3+.
- <strong>Table must be accessed by name:</strong> Iceberg metadata generation is only triggered when the table is accessed by name (not by path).

</div>

---

<div style="background:#E8F5E9;border-left:4px solid #2E7D32;padding:14px 20px;border-radius:4px;margin-top:8px;font-size:15px;color:#1B5E20;line-height:1.9;">
  <strong>🔗 How this connects to Delta Sinks:</strong> Since Streaming Tables and Materialized Views can't use UniForm directly, the pattern in this lecture is to use a <strong>Delta Sink</strong> — write your streaming pipeline output to a plain external Delta table, enable UniForm on that table, and Iceberg clients can immediately read it. This is the recommended production pattern for cross-platform Iceberg access from a live streaming pipeline.
</div>


## D. Connecting the Concepts

These three concepts work together to solve complex real-world data pipeline requirements.Each concept solves a distinct problem. Together they form a complete pattern for pipelines that need real-time processing and cross-platform access.


<div class="mermaid" style="min-width:800px;">
flowchart LR
    C1[" <b>Multiplex</b>\nOne source, many event types\nIngest once — fan out by type"]
    C2[" <b>Delta Sink</b>\nExternal Delta table\ndp.create_sink + append_flow"]
    C3[" <b>Delta UniForm</b>\nExternal tools need access\nAuto-generate Iceberg metadata"]
    C1 -->|"Silver table\nfeeds the sink"| C2
    C2 -->|"Plain Delta table\nenables UniForm"| C3
    style C1 fill:#FF5722,color:#fff,stroke:#E64A19,stroke-width:2px
    style C2 fill:#1565C0,color:#fff,stroke:#0D47A1,stroke-width:2px
    style C3 fill:#2E7D32,color:#fff,stroke:#1B5E20,stroke-width:2px
    linkStyle 0 stroke:#FF7043,stroke-width:2px
    linkStyle 1 stroke:#1976D2,stroke-width:2px
</div>
</div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";
mermaid.initialize({
    startOnLoad: true,
    theme: "default",
    flowchart: { padding: 40, nodeSpacing: 80, rankSpacing: 120 }
});
</script>


#### EXPAND FOR ADDITIONAL DETAILS
<details>

### Combined Advanced Pipeline Pattern

#### 1. Multiplex Pattern
- **Problem**: Multiple event types in one source
- **Solution**: Single ingestion with type-based fan-out
- **Output**: Domain-specific silver tables

#### 2. Delta Sink
- **Problem**: Need external table for cross-platform access
- **Solution**: `dp.create_sink()` + `@dp.append_flow`
- **Output**: Plain Delta table outside pipeline scope

#### 3. Delta UniForm
- **Problem**: External tools need Iceberg format access  
- **Solution**: Auto-generate Iceberg metadata on plain Delta table
- **Output**: Dual-protocol table accessible by any platform
</details>


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>